# Companion notebook — `src/data/acquisition.py`

**Graduated:** `download_nass_yields` (Week 1). `download_weather_data` is still a stub
and is *not* explained here yet.

Runs fully offline on the bundled sample at `data/samples/nass_quickstats_sample.json`
— no API key, no network.

## 1. Motivation

This repo's *volume track* needs thousands of county-year yield observations to make
hierarchical partial pooling and convergence diagnostics visible (Phase 2, milestone
M2a). The USDA NASS **Quick Stats** API provides exactly that: annual, county-level
survey yields for corn and soybeans back past 1990.

The production platform this repo trains for (keragita-farm-intelligence) tags every
record with its `data_source` (its INV-006 — *no untagged data path exists*). This
module mirrors that habit from the very first byte we ingest: every row it writes
carries `data_source = "api_nass"`. Governance card:
`docs/data_cards/DC-usda-nass-quickstats.md`.

## 2. The data semantics (this module's "mathematics")

No formulas here — acquisition is about *provenance semantics*, and getting these wrong
poisons every model downstream:

- **Query anatomy.** Quick Stats is one giant filterable table. Our filters:
  `commodity_desc` (CORN/SOYBEANS), `year__GE` (inclusive start year),
  `statisticcat_desc=YIELD`, `agg_level_desc=COUNTY`, and `source_desc=SURVEY` —
  the annual survey, **not** the 5-yearly census, so years are comparable.
- **Units.** Yields arrive as `BU / ACRE` strings. We keep `unit_desc` in the table;
  conversion to t/ha is a *processing* step (with its own tests), never done silently
  at acquisition.
- **Suppression.** Small counties are disclosure-suppressed: `Value` is `"(D)"`.
  Those become `NaN` — a *missing-data indicator*, not a dropped row (LINV-004).
- **Thousands separators.** NASS renders `1234` as `"1,234"`; parsing must strip
  commas or silently coerce valid data to NaN.

## 3. The shipped implementation

Displayed live from `src/` — never a copy. Three functions: build the URL, fetch the
JSON (all failure modes mapped to the documented `ConnectionError`), tidy + tag.

In [ ]:
import inspect

from IPython.display import Code

from src.data.acquisition import _build_query_url, _records_to_frame, download_nass_yields

Code(
    inspect.getsource(_build_query_url)
    + inspect.getsource(_records_to_frame)
    + inspect.getsource(download_nass_yields),
    language="python",
)

Walkthrough of the load-bearing lines:

- `_build_query_url` never logs: the URL embeds the API key, so the calling code logs
  crop/year only. The key comes exclusively from the `NASS_API_KEY` environment
  variable — never a file, never committed.
- `_records_to_frame` raises `ConnectionError` when the payload has no `data` list —
  that is how Quick Stats signals bad keys, oversized queries (>50k records), and
  malformed filters.
- `pd.to_numeric(..., errors="coerce")` after comma-stripping implements the
  suppression-to-NaN rule from Section 2.

## 4. Worked example (offline, on the bundled sample)

Five real-shaped records: two clean Iowa counties, a comma-separated Illinois value,
a suppressed `(D)` combined-counties row, and a Kansas dryland yield.

In [ ]:
import json
from pathlib import Path

SAMPLE = Path("../../data/samples/nass_quickstats_sample.json")
payload = json.loads(SAMPLE.read_text(encoding="utf-8"))
frame = _records_to_frame(payload)
frame

In [ ]:
# The two provenance invariants in one assertion each: every row is tagged, and
# suppression became NaN instead of disappearing.
assert (frame["data_source"] == "api_nass").all()
assert frame["yield_value"].isna().sum() == 1
frame[["county_name", "yield_value"]]

## 5. Pitfalls and connections

- **The 50,000-record limit.** Quick Stats rejects oversized queries with an error
  payload. Pull per-crop (as the function does) and, if a pull ever grows past the
  limit, split by year range or state.
- **Survey ≠ census.** Mixing `source_desc` values would put 5-yearly census points
  in an annual series — a silent seasonality bug in the yield model later.
- **Suppressed counties are informative.** `(D)` rows mark small production — dropping
  them would bias county-level pooling. They stay as NaN for the imputation week.
- **Connection to Phase 2.** The `year × county` table this produces is exactly the
  long format the hierarchical model (M2a) pools over; `state_alpha`/`county_ansi`
  become the grouping factors.
- **Production parallel.** The same tag-at-ingest discipline appears in
  keragita-farm-intelligence's weather adapters (CHIRPS/NASA POWER), which the Kilifi
  capstone (Phase 4) will mirror.

---
**Provenance:** module `src/data/acquisition.py` @ 0.1.0 · deterministic (no RNG) ·
written 2026-07-07 · author Desmond Momanyi Mariita